# Leyes térmicas de horizonte: ¿fija la cosmología la escala de las galaxias?

**Proyecto:** *Vacuum Particle Journey*, línea A (mecanismo).

**Lo que dijo el notebook anterior.** La ley de Milgrom (1999), sacada de la temperatura de Unruh en un universo con energía oscura, tiene la forma correcta en gravedad débil pero pierde frente a la RAR por ΔBIC = 569, y ninguna escala cosmológica fija funcionaba con ella. El motivo: **no se apaga cuando la gravedad es fuerte**; añade siempre una aceleración extra ≈ s.

**La pista.** La RAR empírica, $a = g_N/(1 - e^{-x})$ con $x = \sqrt{g_N/s}$, contiene el factor $1/(1-e^{-x}) = 1 + n_{BE}(x)$, donde $n_{BE} = 1/(e^x - 1)$ es la ocupación de **Bose-Einstein**: la forma de un baño térmico, que se apaga exponencialmente cuando la "energía" domina sobre la "temperatura".

## Preregistro (escrito antes de calcular nada de este notebook)

**Cuatro leyes**, todas con el mismo límite de gravedad débil ($a \to \sqrt{s\,g_N}$, que da curvas planas y Tully–Fisher) y distinto comportamiento en gravedad fuerte:

| Ley | Fórmula | Gravedad fuerte |
|---|---|---|
| Milgrom 1999 (temperatura de horizonte) | $a = \sqrt{g_N^2 + 2s\,g_N}$ | no se apaga ($a - g_N \to s$) |
| "Simple" de MOND | $a = \tfrac{g_N}{2}\big(1 + \sqrt{1 + 4s/g_N}\big)$ | no se apaga ($a - g_N \to s$) |
| **Bose (térmica)** | $a = g_N\,\big[1 + n_{BE}(x)\big] = \dfrac{g_N}{1 - e^{-x}}$ | se apaga exponencialmente |
| Boltzmann (térmica clásica) | $a = g_N\,\big[1 + e^{-x}/x\big]$ | se apaga exponencialmente |

**Escalas fijadas por la cosmología** (sin ajustar):
- **Hipótesis principal:** $s = a_\Lambda/2\pi$, con $a_\Lambda = c\sqrt{\Lambda/3}$. Es la aceleración cuya energía cuántica natural, $\hbar s/c$, iguala la temperatura del horizonte cósmico, $k_B T_{GH} = \hbar a_\Lambda/2\pi c$.
- Alternativas: $cH_0/2\pi$ (Ho, Minic y Ng) y $cH_0/6$ (Verlinde), con $H_0$ de Planck. La principal también con $H_0$ de SH0ES.

**Aviso de transparencia:** la ley de Bose es exactamente la RAR con $g_\dagger = s$. En el notebook anterior, el intervalo de $g_\dagger$ ajustado (8.3–10.0·10⁻¹¹) ya contenía $a_\Lambda/2\pi$ = 8.6·10⁻¹¹. Así que el ajuste con todas las galaxias **no es una predicción a ciegas**. Las pruebas a ciegas son las tres siguientes.

**Criterios de éxito para la hipótesis principal** (Bose con $s = a_\Lambda/2\pi$), los cuatro a la vez:
1. **Grupo reservado** (misma división por hash que la validación): sin ajustar nada, predice las galaxias reservadas igual o mejor que una RAR cuya escala se ajustó con el grupo de descubrimiento (Δχ² total ≥ −10 y Wilcoxon de una cola, "la ley fija es peor", con p > 0.05). *Corregido antes de usar datos reales: con dos colas, predecir significativamente mejor contaba como fallo.*
2. **Robustez:** con la masa estelar un 20 % mayor o menor (Υ = 0.4 o 0.6), la escala ajustada sigue siendo compatible con $a_\Lambda/2\pi$ dentro de un factor 1.3.
3. **Sistema Solar:** la aceleración anómala que predice en la órbita de Saturno es menor que 10⁻¹⁴ m/s². Es un límite conservador; las efemérides planetarias restringen anomalías de ese orden o menores.
4. **Comparación entre leyes:** con la escala ajustada, las leyes que se apagan (Bose o Boltzmann) ganan a las que no (Milgrom, simple) por ΔBIC > 10.

In [ ]:
import os, io, re, glob, zipfile, urllib.request, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar
from scipy.special import i0, i1, k0, k1
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
RNG = np.random.default_rng(42)
QUICK = False

COL = {"Datos": "#222222", "Solo bariones": "#8a8a85", "NFW ΛCDM (c–M)": "#6250d6",
       "RAR (g† = 1.2e-10)": "#e87ba4", "RAR local (g† global)": "#eb6834", "RAR no local (ER)": "#1baf7a", "Instrumental": "#2a78d6", "Densidad": "#eda100", "Tamaño": "#e34948"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2, "font.size": 10, "legend.frameon": False,
})

G = 4.30091e-6
C_KMS = 299792.458
H0 = 70.0; H0_KPC = H0 / 1000.0
KPC_M = 3.0857e19
TO_MS2 = 1e6 / KPC_M                    # (km/s)^2/kpc -> m/s^2
try:
    import plotly.graph_objects as go
    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False
    print("plotly no disponible: las figuras 3D se harán con matplotlib (estáticas)")
from scipy import fft as sfft
N_GRID = 128
H0_PLANCK, H0_SH0ES, OMEGA_L = 67.4, 73.0, 0.685
C_MS = 2.99792458e8; MPC_M = 3.0857e22
TO_MS2 = 1e6 / 3.0857e19
def cH(H0): return C_MS * H0 * 1e3 / MPC_M
A_L = lambda H0: cH(H0) * np.sqrt(OMEGA_L)
FIXED = {"a_Λ/2π (Planck)": A_L(H0_PLANCK) / (2 * np.pi), "a_Λ/2π (SH0ES)": A_L(H0_SH0ES) / (2 * np.pi),
         "cH₀/2π (Planck)": cH(H0_PLANCK) / (2 * np.pi), "cH₀/6 (Planck)": cH(H0_PLANCK) / 6}
for k, v in FIXED.items(): print(f"  s = {k:18s} = {v:.3e} m/s²")

## 1. Datos y división (la misma de la validación independiente)

In [ ]:
DATA_DIR = "sparc_rotmod"
BASES = ["https://astroweb.case.edu/SPARC/", "http://astroweb.cwru.edu/SPARC/"]
ZENODO = "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1"

def _get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req, timeout=60).read()

def _files():
    return sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_rotmod.dat"), recursive=True))

def get_sparc():
    os.makedirs(DATA_DIR, exist_ok=True)
    if _files():
        return True
    for url in [b + "Rotmod_LTG.zip" for b in BASES] + [ZENODO]:
        try:
            print("Descargando", url)
            zipfile.ZipFile(io.BytesIO(_get(url))).extractall(DATA_DIR)
            if _files():
                return True
        except Exception as e:
            print("  falló:", type(e).__name__)
    try:
        from google.colab import files
        print("Sube Rotmod_LTG.zip (de astroweb.case.edu/SPARC)")
        for name, content in files.upload().items():
            zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
        return bool(_files())
    except Exception:
        return False

def get_quality():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1")
            q = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    q[t[0]] = dict(Q=int(t[17]), Inc=float(t[5]))
            if len(q) > 100:
                return q
        except Exception:
            pass
    return None

def load_rotmod(path):
    name = os.path.basename(path).replace("_rotmod.dat", "")
    d = np.atleast_2d(np.loadtxt(path, comments="#"))
    g = dict(name=name, r=d[:, 0], vobs=d[:, 1], err=d[:, 2], vgas=d[:, 3], vdisk=d[:, 4], vbul=d[:, 5])
    ok = (g["r"] > 0) & (g["err"] > 0)
    o = np.argsort(g["r"][ok])
    for k in ["r", "vobs", "err", "vgas", "vdisk", "vbul"]:
        g[k] = g[k][ok][o]
    g["vgas2"] = g["vgas"] * np.abs(g["vgas"])
    g["vstar2"] = g["vdisk"] * np.abs(g["vdisk"]) + 1.4 * g["vbul"] * np.abs(g["vbul"])
    return g

def _v2_expdisk(r, M, Rd):
    y = r / (2 * Rd)
    return 4 * np.pi * G * (M / (2 * np.pi * Rd**2)) * Rd * y**2 * (i0(y) * k0(y) - i1(y) * k1(y))

def _v2_nfw(r, V200, c):
    x = r / (V200 / (10 * H0_KPC))
    mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))

def make_synthetic_set(n=30, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for name in ["NGC3198", "NGC2403", "DDO154"] + [f"SYN{i:03d}" for i in range(n - 3)]:
        V200 = 10 ** rng.uniform(1.6, 2.4); c = 10 ** rng.normal(1.0, 0.12)
        Rd = rng.uniform(1, 5); Ml = 10 ** rng.uniform(8.5, 11); Mg = Ml * 10 ** rng.uniform(-1, 0.7)
        r = np.linspace(0.3 * Rd, rng.uniform(5, 9) * Rd, rng.integers(12, 40))
        vdisk = np.sqrt(_v2_expdisk(r, Ml, Rd)); vgas = np.sqrt(_v2_expdisk(r, Mg, 2.5 * Rd))
        vtrue = np.sqrt(vgas**2 + 0.5 * vdisk**2 + _v2_nfw(r, V200, c))
        err = 2 + 0.04 * vtrue
        os.makedirs("synthetic", exist_ok=True)
        path = os.path.join("synthetic", f"{name}_rotmod.dat")
        np.savetxt(path, np.c_[r, vtrue + rng.normal(0, err), err, vgas, vdisk, 0 * r], header="Distance = 10 Mpc")
        out[name] = load_rotmod(path)
    return out

HAVE_SPARC = get_sparc()
if HAVE_SPARC:
    GAL = {}
    for f in _files():
        g = load_rotmod(f)
        if len(g["r"]) >= 5:
            GAL[g["name"]] = g
    Q = get_quality()
    if Q:
        GAL = {k: v for k, v in GAL.items() if k in Q and Q[k]["Q"] < 3 and Q[k]["Inc"] >= 30}
        print("Corte de calidad aplicado (Q<3, inc≥30°).")
    else:
        print("Sin tabla de calidad: se usan todas las galaxias.")
else:
    print("⚠️  SIN DATOS REALES: galaxias SINTÉTICAS con halo NFW. Los resultados no tienen valor físico.")
    GAL = make_synthetic_set()

if QUICK and len(GAL) > 40:
    keep = [k for k in ["NGC3198", "NGC2403", "DDO154"] if k in GAL]
    rest = [k for k in GAL if k not in keep]
    GAL = {k: GAL[k] for k in keep + list(RNG.choice(rest, 40 - len(keep), replace=False))}
N_TOT = sum(len(g["r"]) for g in GAL.values())
print(f"Galaxias: {len(GAL)} | puntos: {N_TOT}")

In [ ]:
import hashlib
from scipy.stats import wilcoxon
SPLIT_SALT = "vpj-validacion-2026"
NAMES = list(GAL)
GROUP = np.array(["descubrimiento" if int(hashlib.md5((SPLIT_SALT + k).encode()).hexdigest(), 16) % 2 == 0 else "reservada" for k in NAMES])
DISC, HOLD = np.where(GROUP == "descubrimiento")[0], np.where(GROUP == "reservada")[0]
print(f"Descubrimiento: {len(DISC)} | Reservada: {len(HOLD)}")

## 2. Las cuatro leyes y la tabla de χ²

Para cada ley, cada escala de una rejilla y cada galaxia se busca la mejor masa estelar Υ (con un prior centrado en `U0`, 0.5 por defecto) y se guarda el χ², con 5 % de error sistemático como en los notebooks anteriores.

In [ ]:
def law(gN, s, kind):
    gN = np.clip(gN, 1e-16, None); x = np.sqrt(gN / s)
    if kind == "Milgrom 1999": return np.sqrt(gN**2 + 2 * s * gN)
    if kind == "simple (MOND)": return 0.5 * gN * (1 + np.sqrt(1 + 4 * s / gN))
    if kind == "Bose (térmica)": return gN / (-np.expm1(-x))
    if kind == "Boltzmann (térmica clásica)": return gN * (1 + np.exp(-x) / x)
LAWS = ["Milgrom 1999", "simple (MOND)", "Bose (térmica)", "Boltzmann (térmica clásica)"]
SC = np.geomspace(1e-11, 3e-9, 120)
U_G = np.geomspace(0.1, 1.5, 40)

def table(kind, u0=0.5, floor=0.05):
    pri = (np.log10(U_G / u0) / 0.1) ** 2
    T = np.zeros((len(NAMES), len(SC))); Dd = np.zeros_like(T); npts = 0
    for n, k in enumerate(NAMES):
        g = GAL[k]; r = g["r"]; err = np.sqrt(g["err"] ** 2 + (floor * g["vobs"]) ** 2); npts += len(r)
        gN = (g["vgas2"][None, :] + U_G[:, None] * g["vstar2"][None, :]) / r * TO_MS2
        v = np.sqrt(np.clip(law(gN[:, None, :], SC[None, :, None], kind) * r / TO_MS2, 0, None))
        c = (((g["vobs"] - v) / err) ** 2).sum(-1); t = c + pri[:, None]; i = t.argmin(0)
        T[n] = t[i, np.arange(len(SC))]; Dd[n] = c[i, np.arange(len(SC))]
    return dict(tot=T, dat=Dd, npts=npts)

def at(tab, s, which="dat"):
    j = np.clip(np.searchsorted(SC, s), 1, len(SC) - 1); w = (np.log(s) - np.log(SC[j - 1])) / (np.log(SC[j]) - np.log(SC[j - 1]))
    return tab[which][:, j - 1] * (1 - w) + tab[which][:, j] * w

t0 = time.time()
TABS = {(kind, u0): table(kind, u0) for kind in LAWS for u0 in [0.4, 0.5, 0.6]}
print(f"12 tablas en {time.time()-t0:.0f} s")
NG = len(NAMES); NP = TABS[(LAWS[0], 0.5)]["npts"]

## 3. Todas las galaxias: escala ajustada y escalas cosmológicas fijas

In [ ]:
rows = []
for kind in LAWS:
    tab = TABS[(kind, 0.5)]
    S = tab["tot"].sum(0); j = int(np.argmin(S)); chi = tab["dat"][:, j].sum()
    bs = [SC[np.argmin(tab["tot"][RNG.integers(0, NG, NG)].sum(0))] for _ in range(500)]
    rows.append(dict(ley=kind, escala="ajustada", s=f"{SC[j]:.2e}", IC68=f"{np.percentile(bs,16):.2e}–{np.percentile(bs,84):.2e}",
                     chi2=round(chi), BIC=round(chi + (NG + 1) * np.log(NP))))
    for nm, s in FIXED.items():
        chi = at(tab, s).sum()
        rows.append(dict(ley=kind, escala=nm, s=f"{s:.2e}", IC68="fija", chi2=round(chi), BIC=round(chi + NG * np.log(NP))))
T3 = pd.DataFrame(rows); T3["ΔBIC"] = T3.BIC - T3.BIC.min()
print(T3.to_string(index=False))

### Dónde se diferencian las leyes

Mediana del residuo $\log(g_{obs}/g_{modelo})$ por intervalos de aceleración newtoniana, con cada ley en su escala ajustada. Una ley perfecta es una línea plana en 0. Las leyes que no se apagan deberían desviarse en la zona de gravedad fuerte (derecha).

In [ ]:
bins = np.linspace(-12, -8.5, 15); mid = 0.5 * (bins[1:] + bins[:-1])
cols = {"Milgrom 1999": "#eb6834", "simple (MOND)": "#eda100", "Bose (térmica)": "#1baf7a", "Boltzmann (térmica clásica)": "#2a78d6"}
fig, ax = plt.subplots(figsize=(8.5, 4))
for kind in LAWS:
    tab = TABS[(kind, 0.5)]; j = int(np.argmin(tab["tot"].sum(0))); s = SC[j]
    xs, rs = [], []
    for n, k in enumerate(NAMES):
        g = GAL[k]; r = g["r"]
        pri = (np.log10(U_G / 0.5) / 0.1) ** 2
        gN = (g["vgas2"][None, :] + U_G[:, None] * g["vstar2"][None, :]) / r * TO_MS2
        a = law(gN, s, kind); err = np.sqrt(g["err"] ** 2 + (0.05 * g["vobs"]) ** 2)
        v = np.sqrt(np.clip(a * r / TO_MS2, 0, None)); i = int(np.argmin((((g["vobs"] - v) / err) ** 2).sum(1) + pri))
        ok = (g["vobs"] > 0) & (gN[i] > 0)
        xs.append(np.log10(gN[i][ok])); rs.append(np.log10((g["vobs"][ok] ** 2 / r[ok] * TO_MS2) / a[i][ok]))
    xs, rs = np.concatenate(xs), np.concatenate(rs)
    med = [np.median(rs[(xs >= lo) & (xs < hi)]) if np.sum((xs >= lo) & (xs < hi)) > 15 else np.nan for lo, hi in zip(bins[:-1], bins[1:])]
    ax.plot(mid, med, "o-", ms=4, color=cols[kind], label=kind)
ax.axhline(0, color="#c3c2b7", lw=1)
ax.set_xlabel("log10 g_N [m/s²]  (derecha = gravedad fuerte)"); ax.set_ylabel("log10 (g_obs / g_modelo)")
ax.set_title("Residuo por zona de aceleración (cada ley en su escala ajustada)", loc="left"); ax.legend(fontsize=8.5)
plt.show()

## 4. Prueba 1 (a ciegas): el grupo reservado

La RAR de referencia ajusta su escala **solo con el grupo de descubrimiento**. Cada ley con escala cosmológica fija no ajusta nada. Se comparan galaxia a galaxia en el grupo reservado: $\Delta\chi^2_i = \chi^2_i(\text{RAR ajustada}) - \chi^2_i(\text{ley fija})$, positivo = gana la ley fija.

In [ ]:
ref = TABS[("Bose (térmica)", 0.5)]                       # Bose = RAR
j_ref = int(np.argmin(ref["tot"][DISC].sum(0))); s_ref = SC[j_ref]
chi_ref = ref["dat"][:, j_ref]
print(f"RAR ajustada con el grupo de descubrimiento: g† = {s_ref:.3e} m/s²")
rows = []
for kind in LAWS:
    for nm, s in FIXED.items():
        d = (chi_ref - at(TABS[(kind, 0.5)], s))[HOLD]
        p1 = wilcoxon(d, alternative="less").pvalue if np.any(d != 0) else 1.0
        rows.append(dict(ley=kind, escala=nm, Δχ2_total=round(d.sum(), 1), mediana=round(float(np.median(d)), 2),
                         galaxias_mejor=f"{np.mean(d > 0):.0%}", p_peor=f"{p1:.1e}"))
T4 = pd.DataFrame(rows)
print("Grupo reservado (positivo = la ley con escala fija predice mejor que la RAR ajustada; p_peor pequeño = significativamente peor):")
print(T4.to_string(index=False))

## 5. Prueba 2: robustez frente a la masa estelar

In [ ]:
rows = []
for u0 in [0.4, 0.5, 0.6]:
    for kind in ["Bose (térmica)", "Boltzmann (térmica clásica)"]:
        tab = TABS[(kind, u0)]; s = SC[int(np.argmin(tab["tot"].sum(0)))]
        rows.append(dict(Υ_prior=u0, ley=kind, s_ajustada=f"{s:.2e}", cociente_con_aΛ_2π=round(s / FIXED["a_Λ/2π (Planck)"], 2),
                         a_Λ_dividido_por=round(A_L(H0_PLANCK) / s, 2)))
T5 = pd.DataFrame(rows); print(T5.to_string(index=False))
print(f"(2π = {2*np.pi:.2f})")

## 6. Prueba 3: el Sistema Solar

Cualquier ley tiene que dejar intacta la gravedad donde es fuerte. Calculamos la aceleración anómala $a - g_N$ que predice cada ley, con $s = a_\Lambda/2\pi$, en la órbita de la Tierra y en la de Saturno.

In [ ]:
GM_SUN = 1.32712e20; AU = 1.495978707e11
rows = []
for kind in LAWS:
    s = FIXED["a_Λ/2π (Planck)"]
    r_ = {}
    for nm, d in [("Tierra (1 UA)", 1.0), ("Saturno (9.5 UA)", 9.54)]:
        gN = GM_SUN / (d * AU) ** 2
        if kind in ("Bose (térmica)", "Boltzmann (térmica clásica)"):
            x = np.sqrt(gN / s)
            lg = np.log10(gN) - x / np.log(10) - (0 if kind.startswith("Bose") else np.log10(x))   # log10(g_N e^-x [/x])
        else:
            lg = np.log10(float(law(np.array([gN]), s, kind)[0] - gN))
        r_[nm] = lg
    fmt = lambda lg: f"10^{lg:.0f} m/s²" if lg < -30 else f"{10**lg:.1e} m/s²"
    rows.append(dict(ley=kind, **{k: fmt(v) for k, v in r_.items()}, pasa_límite=("sí" if r_["Saturno (9.5 UA)"] < -14 else "NO")))
T6 = pd.DataFrame(rows); print(T6.to_string(index=False))
print("(Exponencial: en la órbita de Saturno x = √(g_N/s) ≈ 900, así que e^(−x) es prácticamente cero.)")

## 7. Veredicto (con los criterios del preregistro)

In [ ]:
def veredicto():
    if not HAVE_SPARC:
        print("⚠️  Datos SINTÉTICOS: solo comprueba que el código funciona.\n")
    row = T4[(T4.ley == "Bose (térmica)") & (T4.escala == "a_Λ/2π (Planck)")].iloc[0]
    c1 = row.Δχ2_total >= -10 and float(row.p_peor) > 0.05
    rb = T5[T5.ley == "Bose (térmica)"].cociente_con_aΛ_2π
    c2 = bool(np.all((rb >= 1 / 1.3) & (rb <= 1.3)))
    c3 = T6[T6.ley == "Bose (térmica)"].pasa_límite.iloc[0] == "sí"
    fit = T3[T3.escala == "ajustada"].set_index("ley").BIC
    c4 = min(fit["Bose (térmica)"], fit["Boltzmann (térmica clásica)"]) + 10 < min(fit["Milgrom 1999"], fit["simple (MOND)"])
    print(f"1) Grupo reservado, Bose con s = a_Λ/2π sin ajustar: Δχ² = {row.Δχ2_total:+.1f}, p(peor) = {row.p_peor} → {'✓' if c1 else '✗'}")
    print(f"2) Robustez frente a Υ: s ajustada / (a_Λ/2π) = {', '.join(f'{v:.2f}' for v in rb)} → {'✓' if c2 else '✗'}")
    print(f"3) Sistema Solar: anomalía en Saturno {T6[T6.ley == 'Bose (térmica)'].iloc[0]['Saturno (9.5 UA)']} → {'✓' if c3 else '✗'}")
    print(f"4) Las leyes que se apagan ganan a las que no: ΔBIC = {min(fit['Milgrom 1999'], fit['simple (MOND)']) - min(fit['Bose (térmica)'], fit['Boltzmann (térmica clásica)']):+.0f} → {'✓' if c4 else '✗'}")
    print()
    if c1 and c2 and c3 and c4:
        print("→ LA COSMOLOGÍA FIJA LA ESCALA: una ley térmica con s = a_Λ/2π, sin parámetros libres, predice las galaxias reservadas como la RAR "
              "ajustada, es robusta y respeta el Sistema Solar. Falta derivar la ley y el factor 2π desde el entrelazamiento entre horizontes.")
    elif c1 and c2 and c3:
        print("→ LA ESCALA COSMOLÓGICA FUNCIONA, LA FORMA NO SE DECIDE: s = a_Λ/2π sin ajustar predice las galaxias reservadas y es robusta, "
              "pero los datos no separan las leyes que se apagan de las que no (faltan puntos de gravedad fuerte). El Sistema Solar sí las separa.")
    elif c4 and (c1 or c2):
        print("→ CONVERGENCIA PARCIAL: la forma térmica es la correcta, pero la escala cosmológica no queda fijada con seguridad (ver qué criterio falla).")
    elif c4:
        print("→ LA FORMA TÉRMICA ES LA CORRECTA, LA ESCALA NO: las galaxias piden una ley que se apague, pero s = a_Λ/2π no encaja.")
    else:
        print("→ NO SE CONFIRMA: ni la forma térmica ni la escala cosmológica se distinguen de las alternativas.")

veredicto()

## 8. Cómo encaja con la hipótesis

- La ley de Bose es la RAR. Lo nuevo no es la fórmula sino **dos lecturas**: que el factor $1/(1-e^{-x})$ sea una **ocupación térmica**, y que su escala venga del **horizonte cósmico** ($s = a_\Lambda/2\pi$, la aceleración cuya energía cuántica iguala la temperatura de ese horizonte).
- En el lenguaje de tu proyecto: la temperatura de un horizonte es entrelazamiento (notebook anterior, sección 1). Por debajo de $a_\Lambda$, el entrelazamiento que "ve" cada cuerpo lo fija el horizonte cósmico, compartido por toda la galaxia; por encima, su propio horizonte de aceleración lo domina y el efecto se apaga como un baño térmico frío.
- **Lo que falta para que sea física y no numerología:** derivar el factor de Bose y la variable $x = \sqrt{g_N/s}$ desde el entrelazamiento entre los dos horizontes. Si la derivación existe, la materia oscura aparente de las galaxias saldría con **cero parámetros libres** a partir de Λ.